In [6]:
from jupyter_plotly_dash import JupyterDash

import dash
import dash_leaflet as dl
import dash_core_components as dcc
import dash_html_components as html
import plotly.express as px
import dash_table
from dash.dependencies import Input, Output, State
import base64

import os
import numpy as np
import pandas as pd
from bson.json_util import dumps


from pymongo import MongoClient


from Module import AnimalShelter

username = "username"
password = "password"
animal = AnimalShelter(username, password)


df = pd.DataFrame.from_records(animal.readAll({}))



#########################
# Dashboard Layout / View
#########################
app = JupyterDash('Dash DataTable Only')



image_filename = 'Grazioso Salvare Logo.png' # customer image
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

app.layout = html.Div([
    html.Center(html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()),
        style={
            'width' : '25%',
            'height': '25%',
        }
                        )),
    html.Center(html.B(html.H1('Kristopher Collis'))),
    html.Hr(),
    dcc.RadioItems(
        id='Buttons',
        options=[
            {'label': 'All', 'value': 'All'},
            {'label': 'Water Rescue', 'value': 'wr'},
            {'label': 'Mountain or Wilderness Rescue', 'value': 'mwr'},
            {'label': 'Disaster/Individual Tracking', 'value': 'dit'}
        ],
        value='me'
    ),

    html.Hr(),
    dash_table.DataTable(
        id='datatable-id',
        columns=[
            {"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns
        ],
        data=df.to_dict('records'),

        editable=False,
        filter_action="native",
        sort_action="native",
        sort_mode="native",
        column_selectable=False,
        row_selectable="native",
        row_deletable=False,
        selected_columns=[],
        selected_rows=[],
        page_action="native",
        page_current= 0,
        page_size= 10,
    ),
    html.Br(),
    html.Hr(),
    html.Center("Kristopher Collis"),
            html.Div(className='row',
            style={'display' : 'flex'}, 
            children =[
            html.Div(
                id='graph-id',
                className='col s12 m6'
            ),
            html.Div(
                id = 'map-id', 
                className='col s12 m6'
                    ),
            ]
            ),
    html.Br(),
    html.Hr(),
    html.Center("Kristopher Collis")
    ])

#############################################
# Interaction Between Components / Controller
#############################################
#Update_dashboard for the buttons

    
@app.callback([Output('datatable-id','data'),
               Output('datatable-id','columns')],
              [Input('Buttons', 'value')])
def update_dashboard(value):
### FIX ME Add code to filter interactive data table with MongoDB queries
    if (value == 'All'):
        df = pd.DataFrame.from_records(animal.readAll({}))
    if (value == 'wr'):
        df = pd.DataFrame(list(animal.readAll({"breed" : {"$in": ["Chesa Bay Retr Mix", "Labrador Retriever Mix", "Newfoundland Mix"]}, "sex_upon_outcome" : "Intact Female", "age_upon_outcome_in_weeks": {"$lte": 156, "$gte": 26}})))
    if (value == 'mwr'):
        df = pd.DataFrame(list(animal.readAll({"breed" : {"$in": ["German Shephard", "Alaskan Malamute", "Old English Sheepdog", "Siberian Husky", "Rottweiler"]}, "sex_upon_outcome" : "Intact Male", "age_upon_outcome_in_weeks": {"$lte": 156, "$gte": 26}})))
    if (value == 'dit'):
        df = pd.DataFrame(list(animal.readAll({"breed" : {"$in": ["Doberman Pinscher", "German Shephard", "Golden Retriever", "Bloodhound", "Rottweiler"]}, "sex_upon_outcome" : "Intact Male", "age_upon_outcome_in_weeks": {"$lte": 300, "$gte": 20}})))

        
    columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns]
    data=df.to_dict('records')
        
        
    return (data,columns)
    
# #callback for Piechart
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_viewport_data")])
#fucntion for update_graph
def update_graphs(viewData):
    if viewData is not None:
        dff = pd.DataFrame.from_dict(viewData)
        names = dff['breed'].value_counts().keys().tolist()
        values = dff['breed'].value_counts().tolist()
    
    
        return[
            dcc.Graph(
                figure = px.pie(data_frame = dff,values = values,names = names,
                        color_discrete_sequence = px.colors.sequential.RdBu,width = 800,height = 500
                           )
                )
         ]

#callback for update_map
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_viewport_data"),
     Input('datatable-id', 'selected_rows'),
     Input('datatable-id', 'selected_columns')])

#update_function with variables
def update_map(viewData, selected_rows, selected_columns):
    dff = pd.DataFrame.from_dict(viewData)
        #width, height of map, center of map, and how much zoom do you want for map
    return [dl.Map(style = {'width': '1000px', 'height': '500px'}, center = [30.75,-97.48], zoom = 7,
            children = [dl.TileLayer(id = "base-layer-id"),
                        
                #marker with tool tip and popup
                dl.Marker(position=[(dff.iloc[selected_rows[0],13]), (dff.iloc[selected_rows[0],14])], children=[
                    dl.Tooltip(dff.iloc[selected_rows[0],4]),
                    dl.Popup([
                        html.H4("Animal Name"),
                        html.P(dff.iloc[selected_rows[0],9]),

                    ])
                ])
            ])
        ]


app

        
